#### generate required LSTM model results needed for Figures 4, S3, and S4.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import copy
from scipy.special import rel_entr
import time

# --- Setup Compute Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class SphericalLSTM(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64, num_layers=1):
        super(SphericalLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        normed_hidden = self.layer_norm(lstm_out[:, -1, :])
        raw_pred = self.fc(normed_hidden)
        return raw_pred / torch.norm(raw_pred, p=2, dim=1, keepdim=True)

class SphericalLSTM_WithTime(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64, num_layers=1):
        super(SphericalLSTM_WithTime, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # 2-Layer MLP
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        normed_hidden = self.layer_norm(lstm_out[:, -1, :])
        raw_pred = self.fc(normed_hidden)
        return raw_pred / torch.norm(raw_pred, p=2, dim=1, keepdim=True)


class SphericalForecaster:
    def __init__(self, model, seq_length=12, max_epochs=150, lr=0.005, patience=10, weight_decay=0.0):
        self.seq_length = seq_length
        self.max_epochs = max_epochs
        self.lr = lr
        self.patience = patience
        self.weight_decay = weight_decay
        self.model = model.to(device)
        
        self.criterion = nn.MSELoss() 
        
    def create_sequences(self, X_data, Y_data):
        X_seq, y_seq = [], []
        for i in range(len(X_data) - self.seq_length):
            X_seq.append(X_data[i : i + self.seq_length])
            y_seq.append(Y_data[i + self.seq_length])
        return torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32)

    def fit(self, X_window, Y_window, val_ratio=0.2, batch_size=32):
        for layer in self.model.children():
            if hasattr(layer, 'reset_parameters'): layer.reset_parameters()
                
        X_all, y_all = self.create_sequences(X_window, Y_window)
        split_idx = int(len(X_all) * (1 - val_ratio))
        
        X_train, y_train = X_all[:split_idx].to(device), y_all[:split_idx].to(device)
        X_val, y_val = X_all[split_idx:].to(device), y_all[split_idx:].to(device)
        
        # Mini-Batching via DataLoader
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_val_loss, patience_counter, best_weights = float('inf'), 0, None
        
        for epoch in range(self.max_epochs):
            self.model.train()
            
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                
                loss = self.criterion(self.model(batch_x), batch_y)
                loss.backward()
                optimizer.step()
            
            # Validation Phase
            self.model.eval()
            with torch.no_grad():
                val_loss = self.criterion(self.model(X_val), y_val).item()
                
            scheduler.step(val_loss)
                
            if val_loss < best_val_loss:
                best_val_loss, patience_counter = val_loss, 0
                best_weights = copy.deepcopy(self.model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= self.patience: break
                
        if best_weights is not None: self.model.load_state_dict(best_weights)

    def predict(self, initial_x_seq, steps_ahead, future_exo=None):
        self.model.eval()
        curr_seq = torch.tensor(initial_x_seq, dtype=torch.float32).unsqueeze(0).to(device)
        preds = []
        with torch.no_grad():
            for i in range(steps_ahead):
                next_y = self.model(curr_seq) 
                preds.append(next_y.cpu().numpy()[0])
                if future_exo is not None:
                    exo = torch.tensor(future_exo[i], dtype=torch.float32).to(device)
                    next_x = torch.cat((next_y[0], exo), dim=0).unsqueeze(0).unsqueeze(0)
                else:
                    next_x = next_y.unsqueeze(1)
                curr_seq = torch.cat((curr_seq[:, 1:, :], next_x), dim=1)
        return np.array(preds)

# --- Evaluation Metric & CV ---
def calculate_spherical_errors(predictions, targets):
    y_pred = predictions / np.linalg.norm(predictions, axis=1, keepdims=True)
    y_true = targets / np.linalg.norm(targets, axis=1, keepdims=True)
    dot_products = np.clip(np.sum(y_pred * y_true, axis=1), -1.0, 1.0)
    return np.mean(np.arccos(dot_products))

def rolling_window_cv(X, Y, kappa, forecaster):
    T, target_dim = Y.shape[0], Y.shape[1]
    train_size = int(np.floor(T * kappa))
    has_exo = X.shape[1] > target_dim
    results = []
    
    for m in range(T - train_size, 0, -1):
        train_start, train_end = T - train_size - m, T - m
        X_train, Y_train = X[train_start:train_end], Y[train_start:train_end]
        Y_test = Y[train_end : train_end + m]
        future_exo = X[train_end : train_end + m, target_dim:] if has_exo else None
        
        forecaster.fit(X_train, Y_train, val_ratio=0.2)
        seed_seq = X_train[-forecaster.seq_length:]
        Y_pred = forecaster.predict(seed_seq, steps_ahead=m, future_exo=future_exo)
        
        results.append({"m": m, "Sphere_Divergence": calculate_spherical_errors(Y_pred, Y_test)})
    return pd.DataFrame(results)



### prediction results needed for Figures 4 and S3

In [ ]:
if __name__ == "__main__":
    base_dir = "simulated_datasets_7"
    
    results_dir = "LSTM_7"
    os.makedirs(results_dir, exist_ok=True)

    sample_sizes = [120, 300, 600]
    ar_configs = [1, 2, 3]
    num_replicates = 200
    kappa = 0.9
    period = 12
    seq_len = 20 

    total_start_time = time.time()

    for n in sample_sizes:
        for ar in ar_configs:
            folder_name = f"n_{n}_AR{ar}"
            folder_path = os.path.join(base_dir, folder_name)
            
            if not os.path.exists(folder_path):
                print(f"Skipping {folder_name} - Directory not found.")
                continue
                
            print(f"\n--- Processing Setting: {folder_name} ---")
            setting_results = []
            
            for rep in range(1, num_replicates + 1):
                file_path = os.path.join(folder_path, f"sim_{rep:03d}.csv")
                if not os.path.exists(file_path):
                    continue
                
                # Load and prepare data
                Y_raw = pd.read_csv(file_path, header=None).values
                Y_target = Y_raw / np.linalg.norm(Y_raw, axis=1, keepdims=True)
                X_base = Y_target.copy()
                
                # Periodic embeddings
                time_steps = np.arange(Y_target.shape[0])
                sin_time = np.sin(2 * np.pi * time_steps / period).reshape(-1, 1)
                cos_time = np.cos(2 * np.pi * time_steps / period).reshape(-1, 1)
                X_periodic = np.hstack((Y_target, sin_time, cos_time))
                
                # Dynamically Extract Dimensions
                target_dim = Y_target.shape[1]
                base_input_dim = X_base.shape[1]
                periodic_input_dim = X_periodic.shape[1]
                
                # Initialize Updated Models
                base_forecaster = SphericalForecaster(
                    SphericalLSTM(input_dim=base_input_dim, output_dim=target_dim, hidden_dim=64), 
                    seq_length=seq_len, weight_decay=0.0
                )
                
                periodic_forecaster = SphericalForecaster(
                    SphericalLSTM_WithTime(input_dim=periodic_input_dim, output_dim=target_dim, hidden_dim=64), 
                    seq_length=seq_len, weight_decay=1e-3 
                )
                
                # Run CV
                df_base = rolling_window_cv(X_base, Y_target, kappa, base_forecaster)
                df_periodic = rolling_window_cv(X_periodic, Y_target, kappa, periodic_forecaster)
                
                # Merge and tag replicate
                df_merged = pd.merge(df_base, df_periodic, on='m', suffixes=('_Standard', '_Periodic'))
                df_merged['replicate'] = rep
                setting_results.append(df_merged)
                
                if rep % 10 == 0:
                    print(f"  Processed {rep}/{num_replicates} replicates for {folder_name}")

            # Average results
            if setting_results:
                final_df = pd.concat(setting_results)
                avg_df = final_df.groupby('m').mean().drop(columns=['replicate'])
                output_file = os.path.join(results_dir, f"avg_errors_{folder_name}.csv")
                avg_df.to_csv(output_file)
                print(f"Saved aggregated results to {output_file}")

    elapsed = (time.time() - total_start_time) / 60
    print(f"\nAll simulations complete! Total time: {elapsed:.2f} minutes.")
    print(f"Results saved to: {results_dir}")

### prediction results needed for Figure S4

In [ ]:
if __name__ == "__main__":
    base_dir = "simulated_datasets_48"
    
    results_dir = "LSTM_48"
    os.makedirs(results_dir, exist_ok=True)

    sample_sizes = [120, 300, 600]
    ar_configs = [1, 2, 3]
    num_replicates = 200
    kappa = 0.9
    period = 12
    seq_len = 20 

    total_start_time = time.time()

    for n in sample_sizes:
        for ar in ar_configs:
            folder_name = f"n_{n}_AR{ar}"
            folder_path = os.path.join(base_dir, folder_name)
            
            if not os.path.exists(folder_path):
                print(f"Skipping {folder_name} - Directory not found.")
                continue
                
            print(f"\n--- Processing Setting: {folder_name} ---")
            setting_results = []
            
            for rep in range(1, num_replicates + 1):
                file_path = os.path.join(folder_path, f"sim_{rep:03d}.csv")
                if not os.path.exists(file_path):
                    continue
                
                # Load and prepare data
                Y_raw = pd.read_csv(file_path, header=None).values
                Y_target = Y_raw / np.linalg.norm(Y_raw, axis=1, keepdims=True)
                X_base = Y_target.copy()
                
                # Periodic embeddings
                time_steps = np.arange(Y_target.shape[0])
                sin_time = np.sin(2 * np.pi * time_steps / period).reshape(-1, 1)
                cos_time = np.cos(2 * np.pi * time_steps / period).reshape(-1, 1)
                X_periodic = np.hstack((Y_target, sin_time, cos_time))
                
                # Dynamically Extract Dimensions
                target_dim = Y_target.shape[1]
                base_input_dim = X_base.shape[1]
                periodic_input_dim = X_periodic.shape[1]
                
                # Initialize Updated Models
                base_forecaster = SphericalForecaster(
                    SphericalLSTM(input_dim=base_input_dim, output_dim=target_dim, hidden_dim=64), 
                    seq_length=seq_len, weight_decay=0.0
                )
                
                periodic_forecaster = SphericalForecaster(
                    SphericalLSTM_WithTime(input_dim=periodic_input_dim, output_dim=target_dim, hidden_dim=64), 
                    seq_length=seq_len, weight_decay=1e-3 
                )
                
                # Run CV
                df_base = rolling_window_cv(X_base, Y_target, kappa, base_forecaster)
                df_periodic = rolling_window_cv(X_periodic, Y_target, kappa, periodic_forecaster)
                
                # Merge and tag replicate
                df_merged = pd.merge(df_base, df_periodic, on='m', suffixes=('_Standard', '_Periodic'))
                df_merged['replicate'] = rep
                setting_results.append(df_merged)
                
                if rep % 10 == 0:
                    print(f"  Processed {rep}/{num_replicates} replicates for {folder_name}")

            # Average results
            if setting_results:
                final_df = pd.concat(setting_results)
                avg_df = final_df.groupby('m').mean().drop(columns=['replicate'])
                output_file = os.path.join(results_dir, f"avg_errors_{folder_name}.csv")
                avg_df.to_csv(output_file)
                print(f"Saved aggregated results to {output_file}")

    elapsed = (time.time() - total_start_time) / 60
    print(f"\nAll simulations complete! Total time: {elapsed:.2f} minutes.")
    print(f"Results saved to: {results_dir}")